## This is a testing Resume.ai notebook

In [23]:
# imports

import os
import torch
from dotenv import load_dotenv
from IPython.display import Markdown

load_dotenv()  # looks for a .env file in the current directory and its parents

True

In [24]:
from huggingface_hub import login

hf_token = os.getenv("HR_TOKEN")
login(hf_token, add_to_git_credential=True)

`add_to_git_credential=True` is only supported when a token is passed directly. It is ignored by the browser-based login.


In [25]:
audio_path = os.path.abspath("../data/LBCA.mp3")
print(audio_path)
audio_filename = open(audio_path, "rb")

/Users/rodrigoec/cc/projetos/resume.ai/data/LBCA.mp3


# Models for transcribing
## pierreguillou/whisper-medium-portuguese

In [26]:
import mlx_whisper

def transcribe(model_id, file_path):
    result = mlx_whisper.transcribe(file_path, path_or_hf_repo=model_id, language="pt")
    return result["text"]

In [28]:
mac_whisper = transcribe("mlx-community/whisper-large-v3-turbo", audio_path)

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 459.50it/s]


In [36]:
Markdown(f"{mac_whisper[:300]}...")

 Eu entrei na LBCA no início de abril de 2026 agora e a minha ideia aqui é meio que contar as experiências que eu tive para depois a gente fazer o resumo dessas informações com o IA. Então quando eu entrei na LBCA eu entrei bem como perfil de intermediação DEV com cientista de dados, então eles sabi...

In [ ]:
mac_whisper_smaller = transcribe("mlx-community/whisper-turbo", audio_path)


In [38]:
Markdown(f"{mac_whisper_smaller[:300]}...")


 Eu entrei na LBCA no início de abril de 2026 agora e a minha ideia aqui é meio que contar as experiências que eu tive para depois a gente fazer o resumo dessas informações com o IA. Então quando eu entrei na LBCA eu entrei bem como perfil de intermediação DEV com cientista de dados, então eles sabi...

## Creating structured topics

In [58]:
load_dotenv("../.env")  # looks for a .env file in the current directory and its parents


True

In [65]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


class Topic(BaseModel):
    """Structured topics that describes an experience of a job position using STAR method."""

    company: str = Field(
        description="Company where I worked in on this Topic",
        examples=["LBCA", "ePol", "NuFuturo"],
    )
    situation: str = Field(
        description="The context. What was going on, and what was the problem or need?",
        examples=[
            "Every new client project required rebuilding the document intake logic from scratch."
        ],
    )
    task: str = Field(
        description="your responsibility in that situation. What were you expected to achieve?",
        examples=[
            "Build a reusable service to extract structured data from any document."
        ],
    )
    action: list[str] = Field(
        description="what you specifically did. This is usually the longest part, focused on your decisions, tools, and approach.",
        examples=[
            "Built an OCR + LLM extraction pipeline with AWS Textract and Bedrock, integrated via Kafka."
        ],
    )
    result: str = Field(
        description="the outcome, ideally measurable. What changed because of your work?",
        examples=[
            "The service is now reused across multiple systems, eliminating rework on new projects."
        ],
    )


class JobExperiences(BaseModel):
    """List of job experiences topics"""
    topics: list[Topic] = Field(description="List of STAR topics for the current job")

In [62]:
SYSTEM_PROMPT = """
You are an expert technical resume writer and career coach specializing in software engineering and AI roles. You turn raw, informal accounts of work experience (often voice transcriptions) into resume-ready accomplishments written with the STAR method (Situation, Task, Action, Result).

## Your process
1. Read the whole transcript before writing. Transcriptions are messy: they contain filler words, repetitions, self-corrections, and misheard technical terms.
2. Identify distinct accomplishments. Group related details into one experience. Split unrelated work into separate experiences. Typical categories are products or systems built, architecture decisions, reliability/observability/testing work, leadership, and cross-functional collaboration. Aim for 4–8 experiences, ordered by impact.
3. For each experience, extract the Situation, Task, Action, and Result from what the person actually said.

## Accuracy rules (critical)
- NEVER invent facts, metrics, percentages, team sizes, or outcomes. Use only what is stated or directly implied.
- When a number would strengthen a result but wasn't given, insert a clear placeholder like [X%], [N documents/month], or [Y hours saved] so the person can fill it in.
- Fix obvious transcription errors in technical terms (e.g., "Text Track" → Textract, "Cloud Sonnet" → Claude Sonnet, misspelled company names), and list every correction you made at the end.
- If the person says they led, proposed, or owned something, say so. If they say it was a shared effort, reflect that honestly (e.g., "together with a colleague").
- Keep the person's real technologies and tool names. Don't add technologies they didn't mention.

## Writing rules
- Write in the language specified by the user (default: Brazilian Portuguese).
- Start Action bullets with strong first-person past-tense verbs (e.g., Desenvolvi, Implementei, Propus, Liderei, Estruturei).
- Be specific and technical in Actions: name the tools, patterns, and architectural choices.
- Results should express impact (business value, efficiency, reliability, adoption, reuse), never just "the task was completed."
- Where a concrete example makes an abstract point clearer, add ONE short example in parentheses (e.g., "(ex.: CNPJ, endereços completos)").
- No fluff, no buzzword stacking, no exaggeration.

## Output format (Markdown)
Start with a header line:
**[Role] · [Company]** | [Period]
*[One-line description of the company/context, if inferable]*

Then for each experience:

### N. [Short descriptive title]

- **Situação:** 1–2 sentences on context and the problem.
- **Tarefa:** 1 sentence on what the person needed to achieve.
- **Ação:** concrete bullet points of what the person did.
- **Resultado:** the outcome and impact, with real numbers or [placeholders].

**Versão para o currículo:**
> One single-sentence bullet (max ~40 words) that condenses the experience: action verb + what + how (key tech) + result.

---

After all experiences, add:
1. **Stack:** a single line listing the technologies mentioned.
2. **Pontos para completar antes de usar:** a short list of the placeholders to fill in, plus suggestions for metrics that would strengthen the resume, plus the transcription corrections you made.

Translate the section labels (Situação/Tarefa/Ação/Resultado, etc.) into the output language if it isn't Portuguese."""

In [ ]:
USER_PROMPT = '''
Read this transcript of my experiences in my current job and create Markdown bullet points representing experiences suitable for my resume. Phrase them according to the STAR method.
<transcript>
{TRANSCRIPT}
</transcript>
'''

In [ ]:
def create_topics(model_id, model_provider, region_name=None):
  llm = init_chat_model(
      model_id,
      model_provider=model_provider,
      region_name=region_name,
  )
  structured_llm = llm.with_structured_output(JobExperiences)

  result = structured_llm.invoke([
    ("system", SYSTEM_PROMPT),
    ("user", USER_PROMPT.format(TRANSCRIPT=mac_whisper)),
  ])
  
  return result


In [ ]:
result = create_topics("openai/gpt-oss-120b", model_provider="huggingface")

In [92]:
md_string = ""

for index, topic in enumerate(result.topics):
  
  formated_topic = f"""
  ### Experiência {index + 1} - {topic.situation}
  - **Task**: {topic.task}
  - **Action**: {topic.action}
  - **Result**: {topic.result}
  - **Company**: {topic.company}
  \n
  """
  
  md_string += formated_topic


Markdown(md_string)


  ### Experiência 1 - A LBCA, um escritório de advocacia que usa IA para automação de documentos legais, precisava de um serviço reutilizável para extrair informações estruturadas de documentos. Anteriormente, cada novo projeto de cliente requeria reconstrução completa da lógica de cadastro e extração.
  - **Task**: Desenvolver um produto de cadastro (Standalone/Hub) que extraísse estruturadamente dados de documentos de forma agnóstica, permitindo reutilização em múltiplos projetos e clientes.
  - **Action**: ['Implementei um pipeline de extração de informações que combina extração de texto com OCR usando AWS Textract e análise de imagens via Amazon Bedrock com Claude Sonnet.', 'Estruturei a captura de dados do cliente (nome do campo, descrição/prompt, tipo de entidade) em um JSON fortemente tipado para passar como ferramenta estruturada à IA.', 'Integrei CloudWatch para logging em camadas de eventos, mascarando dados sensíveis e gerando resumos de execução para rastreabilidade.', 'Implementei testes unitários e end-to-end para garantir cobertura acima de 80% e melhorar a confiabilidade do serviço.']
  - **Result**: O serviço Hub tornou-se reutilizável em múltiplos projetos, eliminando reinvenção da roda. A plataforma de cadastro genérica elimina redundância de código entre clientes e projetos, reduzindo tempo de integração.
  - **Company**: LBCA
  

  
  ### Experiência 2 - O projeto Hub precisava permanecer agnóstico e genérico, mas alguns clientes exigiam pós-processamento e extração específicos (ex.: chamadas a APIs externas, campos customizados) que não poderiam ser incorporados ao produto principal sem comprometer sua escalabilidade e manutenibilidade.
  - **Task**: Propor uma arquitetura que mantivesse o Hub agnóstico enquanto permitisse extensões específicas por cliente, com claras responsabilidades de componentes.
  - **Action**: ['Propus (junto com um colega) a arquitetura baseada em Kafka: o Hub consome mensagens de um tópico específico, processa de forma agnóstica e publica em um tópico de saída com status de sucesso ou falha.', 'Estruturei a solução em camadas: Hub (extração genérica) e Hub Plus (produto derivado para cada cliente com requirements específicos), cada um consumindo tópicos Kafka separados.', 'Estabeleci que o Hub Plus reutilizaria o serviço agnóstico como base, agregando pós-processamento, extração de campos customizados e integrações externas por cliente.', 'Documentei a decisão arquitetural para orientar discussões futuras sobre escalabilidade e manutenção do sistema.']
  - **Result**: A arquitetura de camadas permitiu isolamento de complexidade: cada cliente com necessidades específicas contrata e paga pelo Hub Plus, mantendo o Hub simples e confiável. Aplicação clara de engenharia de software em decisões de sistema distribuído.
  - **Company**: LBCA
  

  
  ### Experiência 3 - O projeto de contestação de processos tinha uma arquitetura complexa com múltiplas etapas de extração (alegações iniciais, subsídios, imagens) que executavam sequencialmente, causando latência na geração de documentos para os advogados.
  - **Task**: Identificar oportunidades de paralelização e melhorias arquiteturais para acelerar o pipeline de extração e processamento de regras de teses.
  - **Action**: ['Propus paralelizar as três esteiras de extração (alegações, subsídios, seleção de imagens) que antes executavam sequencialmente.', 'Implementei processamento de imagens via Bedrock para descrever imagens e descrevê-las como texto, preservando contexto e permitindo seleção posterior pela IA.', 'Estruturei lógica de avaliação de teses usando JSON Logic: regras booleanas configuradas no front-end são avaliadas serverside para determinar inclusão/exclusão no documento final.', 'Implementei etapa de interpolação que substitui referências de variáveis nos templates das teses pelos valores extraídos do documento.']
  - **Result**: A paralelização das esteiras reduziu latência do pipeline. O sistema agora processa múltiplos clientes em teste de produção do projeto de contestação. Melhor experiência para advogados com documentos gerados mais rapidamente.
  - **Company**: LBCA
  

  
  ### Experiência 4 - Um bug crítico não foi rastreável pelos logs existentes, revelando cobertura de testes insuficiente (abaixo de 80%). A falta de testes robustos dificultava identificação de regressões e rastreamento de problemas em produção.
  - **Task**: Aumentar a cobertura de testes e confiabilidade do sistema através de testes unitários e end-to-end, garantindo rastreabilidade de bugs futuros.
  - **Action**: ['Implementei estratégia focada em testes unitários de componentes críticos de extração e processamento.', 'Adicionei testes end-to-end do pipeline completo (extração → validação de regras → interpolação) para validar fluxos inteiros.', 'Utilizei discussões com IA (ex.: LLMs) para explorar possíveis cenários de falha e casos extremos durante design de testes.', 'Mantive cobertura de testes como critério contínuo em novas features e decisões arquiteturais.']
  - **Result**: Cobertura de testes aumentada para acima de 80%. Bugs agora são detectados mais rapidamente e rastreáveis via logs estruturados. Confiança aumentada nas mudanças de código.
  - **Company**: LBCA
  

  
  ### Experiência 5 - O projeto de contestação envolvia orientar advogados (equipes não-técnicas) sobre como usar a plataforma e estruturar prompts eficientes para extração de informações legais. Conceitos técnicos (JSON, OCR, LLMs, regras booleanas) eram abstratos para públicos não-técnicos.
  - **Task**: Traduzir conceitos técnicos de IA e engenharia de software em linguagem acessível, guiando advogados no uso eficiente da plataforma e design de prompts.
  - **Action**: ['Desenvolvei habilidade de traduzir conceitos técnicos (estruturas JSON, lógica de regras, processamento de imagens) em analogias e exemplos práticos para advogados.', 'Orientei times de law na configuração de teses, regras de inclusão/exclusão e seleção de fatos com exemplos concretos de casos reais.', 'Facilitei discussões entre desenvolvedores e advogados para alinhar expectativas sobre capacidades e limitações da plataforma.', 'Realizei sesões de treinamento sobre prompt engineering específico para o domínio legal (ex.: estruturar perguntas para extração de CNPJ, endereços, vouchers).']
  - **Result**: Advogados adotaram a plataforma com confiança. Mais de 10 clientes em teste de produção usando a plataforma. Redução de friction entre times técnicos e de law. Feedback melhor estruturado para iterações de produto.
  - **Company**: LBCA
  

  
  ### Experiência 6 - Os projetos de cadastro e contestação precisavam de observabilidade robusta para rastreamento de bugs e troubleshooting em produção. Logs desorganizados dificultavam identificação de falhas e impacto de mudanças.
  - **Task**: Implementar estratégia de logging em eventos estruturada, garantindo rastreabilidade por caso e mascaramento de dados sensíveis.
  - **Action**: ['Estruturei logging em camadas de eventos e sub-eventos, capturando cada etapa do pipeline de extração e processamento com status e erros.', 'Implementei mascaramento de dados sensíveis nos logs para garantir conformidade e privacidade.', 'Adicionei ID de caso/referência em todos os logs para permitir rastreamento de fluxo completo no CloudWatch.', 'Criei resumos de execução ao final de cada processo, consolidando status de todas as etapas e mensagens de erro.']
  - **Result**: Observabilidade significativamente melhorada: troubleshooting agora é rápido e confiável. Rastreamento de bugs por ID de caso. Manutenção e correção de problemas em produção são mais eficientes. [X% de redução no tempo médio de resolução de incidentes].
  - **Company**: LBCA
  

  
  ### Experiência 7 - Você foi adicionado ao time com background de engenharia de software, em um contexto onde era esperado conhecimento equilibrado de data science. O foco inicial era trabalhar em projetos de IA dentro da organização.
  - **Task**: Contribuir como AI-engineer, aplicando engenharia de software e arquitetura robusta em produtos que usam IA para automação de documentos legais.
  - **Action**: ['Começou com sustentação e manutenção de projetos existentes enquanto se aprofundava nos projetos principais (Hub e contestação).', 'Posicionou-se como especialista em engenharia de software para IA (AI-engineer), diferenciando-se de especialistas em prompt engineering ou data science.', 'Liderou (junto com um colega) discussões arquiteturais e decisões de sistema, sempre focando em manutenibilidade, escalabilidade e engenharia sólida.', 'Utilizou IA como ferramenta de apoio a decisões arquiteturais, não como substituto: sempre validando e questionando propostas.']
  - **Result**: Estabeleceu-se como figura central nos projetos, sendo líder de decisões arquiteturais. Reconhecimento por equilibrar engenharia de software rigorous com inovação em IA. Oportunidade de influenciar direção técnica de produtos.
  - **Company**: LBCA
  

  